In [1]:
import cv2
import mediapipe as mp
import numpy as np
import os
import time
import tensorflow as tf
# from tensorflow.keras.models import Model
from collections import deque

from utils import mediapipeDetection, drawStyledLandmarks, extractLandmarks, checkVelocity, faceNormalization, extractFaceLandmarks

Notes:

This section of the code is for the OpenCV window to capture frames of data from the user to then interpret each words signified which can be strung into a sentence.


* predictionContainer - holds the per frame action of the user until enough is collected which can then be used for prediction
      - efficient for time and memory
* wordsPredictedContainer - contains predicted words which can be used for translation at a later point
* confidenceLevel - simple checker value for prediction
* frameLimit - currently each video resource for a clip is 230? frames in length
* actionLmk - 

230 frame limit for testing, ~244 frame limit on video training

In [2]:
def actionLmkAddVelocity(fullData):
    dataCopy = fullData.copy()
    
    velocityArr = np.diff(dataCopy[:,1404:], axis=0)
    padding = np.zeros_like(velocityArr[:1], dtype=np.float32)      # creates a padding based on the shape of one the landmarks of the frame
    velocityArr = np.concatenate((padding, velocityArr), axis=0)    # completed velocity array

    return np.concatenate((dataCopy, velocityArr), axis=1)


convert model.predict into a low level execution for saving ram and efficiency
good evening and good afternoon too similar might need to fix that through epochs? or new layer for smaller context

In [3]:
labels = np.array(["goodMorning","goodAfternoon","goodEvening","hello","howAreYou","imFine","niceToMeetYou","thankYou","youreWelcome","seeYouTomorrow"])
prediction = np.zeros(len(labels))
confidenceLevel = 0.75
frameLimit = 40
frameCount = 0
frameResultsContainer = deque(maxlen=frameLimit)
skipRate = 3

clips = ["test/0/1.mov","test/1/0.mov", "test/2/1.mov", "test/3/1.mov", "test/4/1.mov", "test/5/1.mov", "test/6/0.mov", "test/7/1.mov", "test/8/0.mov", "test/9/1.mov"]
labelNum = 0
cap = cv2.VideoCapture(clips[labelNum])
print(labels[labelNum])

modelAction = tf.keras.models.load_model('modelAction98-96.keras')
# modelAction.summary()

dummy_input = tf.zeros((1, 40, 516))
_ = modelAction(dummy_input, training=False)


mpHolistic = mp.solutions.holistic     #Holistic Model


print("\nStarting inference\n")

actionStart = False
currFrame = None
prevFrame = None

with mpHolistic.Holistic( min_detection_confidence=0.75, min_tracking_confidence=0.75) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("failed to read video")
            continue

        image, results = mediapipeDetection(frame, holistic)
        drawStyledLandmarks(image, results)         # can be commented out if visualization not needed
        actionLmk = extractLandmarks(results)
        
        # New Velocity Check
        if frameCount == 0:
            currFrame = actionLmk

        if frameCount > 0 and not actionStart:
            prevFrame = currFrame
            currFrame = actionLmk

            if not actionStart:
                if checkVelocity(currFrame, prevFrame):
                    actionStart = True
                    print("Action Started at frame:", frameCount)
                    # mainly for testing

        if actionStart and frameCount % skipRate == 0:

            frameResultsContainer.append(actionLmk)     # collects frame here (ensured to be frameLimit due to deque)
            if len(frameResultsContainer) == frameLimit:
                snapshot = actionLmkAddVelocity(np.array(frameResultsContainer))[:, 1404:]

                # prediction = modelAction.predict(np.expand_dims(snapshot, axis=0))[0] # data treatment (40,516) -> (1,40,516)

                # 2. Convert to Tensor (Explicitly tells TF "This is data, not a new code path")
                input_tensor = tf.convert_to_tensor(np.expand_dims(snapshot, axis=0), dtype=tf.float32)

                # 3. FAST INFERENCE (Replaces .predict)
                # training=False is critical! It tells Dropout layers to turn off.
                predictions_tensor = modelAction(input_tensor, training=False)
                
                # 4. Convert back to Numpy to read the answer
                prediction = predictions_tensor.numpy()[0]


                frameResultsContainer.clear()
                print(prediction)
                print(prediction.max())
                print(labels[np.argmax(prediction)])
                # delete cap release here if predictions are now made correctly
                cap.release()
                cv2.destroyAllWindows()
        cv2.imshow('OpenCV Feed', image)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
        
        frameCount += 1

    cap.release()
    cv2.destroyAllWindows()

goodMorning

Starting inference



c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Action Started at frame: 87
[9.9999571e-01 5.8824060e-07 2.9137509e-06 8.7891584e-07 2.2847866e-11
 5.1017662e-11 7.7906726e-10 2.5503395e-13 4.4794046e-08 7.0458652e-09]
0.9999957
goodMorning


In [ ]:
modelMood = tf.keras.models.load_model('moodFold2.keras')
modelMood.summary()

dummy_input = tf.zeros((1,1412,))
_ = modelMood(dummy_input, training=False)


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 1412)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │        90,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mood_output (Dense)             │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 278,149 (1.06 MB)

 Trainable params: 92,673 (362.00 KB)

 Non-trainable params: 128 (512.00 B)

 Optimizer params: 185,348 (724.02 KB)

In [36]:
mood = np.array(["neutral","questioning"])
images = [
    "FaceData/UnseenData/neutral/0.png",
    "FaceData/UnseenData/neutral/1.png",
    "FaceData/UnseenData/neutral/2.png",
    "FaceData/UnseenData/neutral/3.png",
    "FaceData/UnseenData/neutral/4.png",
    "FaceData/UnseenData/question/0.png",
    "FaceData/UnseenData/question/1.png",
    "FaceData/UnseenData/question/2.png",
    "FaceData/UnseenData/question/3.png",
    "FaceData/UnseenData/question/4.png"
]
test = 9
mpHolistic = mp.solutions.holistic     #Holistic Model

with mpHolistic.Holistic(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=0, refine_face_landmarks=False, smooth_landmarks=True) as holistic:
    # I wish I was special
    # you'd go by here
    # just for me

    image = cv2.imread(images[test])
    _, results = mediapipeDetection(image, holistic)
    landmarks = extractFaceLandmarks(results)
    landmarks = faceNormalization(landmarks)

    # 2. Convert to Tensor (Explicitly Tells TF "This is data, not a new code path")
    input_tensor = tf.convert_to_tensor(np.expand_dims(landmarks, axis=0), dtype=tf.float32)
    # 3. FAST INFERENCE (Replaces .predict)
    # training=False is critical! It tells Dropout layers to turn off.
    predictions_tensor = modelMood(input_tensor, training=False)
    # 4. Convert back to Numpy to read the answer
    prediction = predictions_tensor.numpy()[0][0]  # scalar between 0 and 1
    print(prediction)
    predicted_mood = mood[int(prediction > 0.5)]  # threshold at 0.5
    print(predicted_mood)

    confidence = abs(prediction - 0.5) * 2  # rescales 0.5->1.0 range to 0->1
    print(f"Predicted: {predicted_mood} (confidence: {confidence:.2%})")

    

0.47252867
neutral
Predicted: neutral (confidence: 5.49%)
